In [ ]:
import os
import csv

ROOT_DIR = "/home/danny.xie/data/dxie/squeeze_every_bit/src/output/None"
OUTPUT_CSV = "results.csv"


def read_second_number(txt_path):
    with open(txt_path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    if len(lines) >= 2:
        return float(lines[1])
    return None


def parse_experiment_path(path):
    """
    Example:
    .../seed1/9_100/fewshotOOD@swinv2_base_window8_256.ms_in1k@sam3
    """
    parts = path.split(os.sep)

    seed = next(p for p in parts if p.startswith("seed"))
    train_test = parts[parts.index(seed) + 1]  # e.g. 9_100
    method_block = parts[-1]

    train_shots, test_shots = train_test.split("_")

    tokens = method_block.split("@")
    method = tokens[0]

    feature_extractor = None
    sam_proposal = None

    for t in tokens[1:]:
        if "sam" in t:
            sam_proposal = t
        else:
            feature_extractor = t

    return {
        "seed": int(seed.replace("seed", "")),
        "train_shots": int(train_shots),
        "test_shots": int(test_shots),
        "method": method,
        "feature_extractor": feature_extractor,
        "sam_proposal": sam_proposal,
    }


rows = []

for root, _, files in os.walk(ROOT_DIR):
    for file in files:
        if not file.startswith("mAP_") or not file.endswith(".txt"):
            continue

        txt_path = os.path.join(root, file)
        value = read_second_number(txt_path)

        if value is None:
            continue

        meta = parse_experiment_path(root)

        rows.append({
            **meta,
            "metric": file.replace(".txt", ""),
            "value": value,
            "file": file,
        })


with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "seed",
            "train_shots",
            "test_shots",
            "method",
            "feature_extractor",
            "sam_proposal",
            "metric",
            "value",
            "file",
        ],
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {OUTPUT_CSV}")
